In [0]:
# Set the catalog
spark.sql("USE CATALOG final_assesment")

# Set the schema
spark.sql("USE SCHEMA gold")

# Verify the current catalog and schema
print("Current Catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Current Schema:", spark.sql("SELECT current_schema()").first()[0])

In [0]:
# Read the Silver table
df_silver = spark.table("final_assesment.silver.orders_clean")

# Display schema
df_silver.printSchema()

# Display sample records
display(df_silver.limit(10))

# Check total record count
print("Total Silver Records:", df_silver.count())

### Creation of Customer Dimension

In [0]:
# Select unique customer records
df_customer = (
    df_silver
    .select(
        "customer_id",
        "customer_name",
        "customer_city",
        "customer_state",
        "customer_dob"
    )
    .dropDuplicates(["customer_id"])
)

# Check customer count
print("Total Unique Customers:", df_customer.count())

display(df_customer)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS final_assesment.gold.dim_customer (
    customer_key BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING NOT NULL,
    customer_name STRING,
    customer_city STRING,
    customer_state STRING,
    customer_dob DATE,

    CONSTRAINT pk_dim_customer
        PRIMARY KEY (customer_key)
)
USING DELTA;



In [0]:
from delta.tables import DeltaTable

customer_target = DeltaTable.forName(
    spark,
    "final_assesment.gold.dim_customer"
)

(
    customer_target.alias("t")
    .merge(
        df_customer.alias("s"),
        "t.customer_id = s.customer_id"
    )
    .whenMatchedUpdate(set={
        "customer_name": "s.customer_name",
        "customer_city": "s.customer_city",
        "customer_state": "s.customer_state",
        "customer_dob": "s.customer_dob"
    })
    .whenNotMatchedInsert(values={
        "customer_id": "s.customer_id",
        "customer_name": "s.customer_name",
        "customer_city": "s.customer_city",
        "customer_state": "s.customer_state",
        "customer_dob": "s.customer_dob"
    })
    .execute()
)

print("Customer dimension merged successfully!")

In [0]:
%sql
SELECT *
FROM final_assesment.gold.dim_customer
ORDER BY customer_key;

### Creation of Product Dimension

In [0]:
df_product = (
    df_silver
    .select(
        "product_id",
        "product_name",
        "category",
        "brand"
    )
    .dropDuplicates(["product_id"])
)

print("Total Unique Products:", df_product.count())

display(df_product)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS final_assesment.gold.dim_product (
    product_key BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id STRING NOT NULL,
    product_name STRING,
    category STRING,
    brand STRING,

    CONSTRAINT pk_dim_product
        PRIMARY KEY (product_key)
)
USING DELTA;

In [0]:
product_target = DeltaTable.forName(
    spark,
    "final_assesment.gold.dim_product"
)

(
    product_target.alias("t")
    .merge(
        df_product.alias("s"),
        "t.product_id = s.product_id"
    )
    .whenMatchedUpdate(set={
        "product_name": "s.product_name",
        "category": "s.category",
        "brand": "s.brand"
    })
    .whenNotMatchedInsert(values={
        "product_id": "s.product_id",
        "product_name": "s.product_name",
        "category": "s.category",
        "brand": "s.brand"
    })
    .execute()
)

print("Product dimension merged successfully!")

In [0]:
%sql
SELECT *
FROM final_assesment.gold.dim_product
ORDER BY product_key;

### Creation of Payment Dimension

In [0]:
df_payment = (
    df_silver
    .select(
        "payment_method_id",
        "payment_method",
        "payment_provider"
    )
    .dropDuplicates(["payment_method_id"])
)

print("Total Unique Payment Methods:", df_payment.count())

display(df_payment)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS final_assesment.gold.dim_payment (
    payment_key BIGINT GENERATED ALWAYS AS IDENTITY,
    payment_method_id STRING NOT NULL,
    payment_method STRING,
    payment_provider STRING,

    CONSTRAINT pk_dim_payment
        PRIMARY KEY (payment_key)
)
USING DELTA;

In [0]:
payment_target = DeltaTable.forName(
    spark,
    "final_assesment.gold.dim_payment"
)

(
    payment_target.alias("t")
    .merge(
        df_payment.alias("s"),
        "t.payment_method_id = s.payment_method_id"
    )
    .whenMatchedUpdate(set={
        "payment_method": "s.payment_method",
        "payment_provider": "s.payment_provider"
    })
    .whenNotMatchedInsert(values={
        "payment_method_id": "s.payment_method_id",
        "payment_method": "s.payment_method",
        "payment_provider": "s.payment_provider"
    })
    .execute()
)

print("Payment dimension merged successfully!")

In [0]:
%sql
SELECT *
FROM final_assesment.gold.dim_payment
ORDER BY payment_key;

### Create of Date Dimension

In [0]:
from pyspark.sql.functions import (
    col,
    min,
    max,
    sequence,
    explode,
    date_format,
    year,
    quarter,
    month,
    dayofmonth,
    dayofweek,
    dayofyear,
    weekofyear,
    date_format,
    when,
    lit
)

# Find minimum and maximum order dates
date_range = df_silver.agg(
    min("order_date").alias("min_date"),
    max("order_date").alias("max_date")
).first()

min_date = date_range["min_date"]
max_date = date_range["max_date"]

print("Minimum Order Date:", min_date)
print("Maximum Order Date:", max_date)

# Generate all dates in the range
df_dates = spark.sql(f"""
    SELECT explode(
        sequence(
            DATE('{min_date}'),
            DATE('{max_date}'),
            INTERVAL 1 DAY
        )
    ) AS full_date
""")

display(df_dates)

In [0]:
df_dim_date = (
    df_dates
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("full_date")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
    .withColumn(
        "is_weekend",
        when(dayofweek(col("full_date")).isin(1, 7), True).otherwise(False)
    )
    .select(
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day_name",
        "is_weekend"
    )
)

display(df_dim_date)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS final_assesment.gold.dim_date (
    date_key INT NOT NULL,
    full_date DATE,
    year INT,
    quarter INT,
    month INT,
    month_name STRING,
    day_name STRING,
    is_weekend BOOLEAN,

    CONSTRAINT pk_dim_date
        PRIMARY KEY (date_key)
)
USING DELTA;

In [0]:
df_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_assesment.gold.dim_date")

In [0]:
%sql
SELECT *
FROM final_assesment.gold.dim_date
ORDER BY full_date;

### Creation of Fact Table

In [0]:
from pyspark.sql.functions import col

# Load the four dimension tables
df_dim_customer = spark.table("final_assesment.gold.dim_customer")
df_dim_product = spark.table("final_assesment.gold.dim_product")
df_dim_payment = spark.table("final_assesment.gold.dim_payment")
df_dim_date = spark.table("final_assesment.gold.dim_date")

# Display dimension counts
print("Customer Dimension:", df_dim_customer.count())
print("Product Dimension:", df_dim_product.count())
print("Payment Dimension:", df_dim_payment.count())
print("Date Dimension:", df_dim_date.count())

In [0]:
df_fact = (
    df_silver.alias("s")

    # Join Date Dimension
    .join(
        df_dim_date.alias("d"),
        col("s.order_date") == col("d.full_date"),
        "left"
    )

    # Join Customer Dimension
    .join(
        df_dim_customer.alias("c"),
        col("s.customer_id") == col("c.customer_id"),
        "left"
    )

    # Join Product Dimension
    .join(
        df_dim_product.alias("p"),
        col("s.product_id") == col("p.product_id"),
        "left"
    )

    # Join Payment Dimension
    .join(
        df_dim_payment.alias("pm"),
        col("s.payment_method_id") == col("pm.payment_method_id"),
        "left"
    )

    # Select fact table columns
    .select(
        col("s.order_id"),
        col("d.date_key"),
        col("c.customer_key"),
        col("p.product_key"),
        col("pm.payment_key"),

        col("s.order_status"),
        col("s.order_updated_at"),

        col("s.quantity"),
        col("s.unit_price"),
        col("s.discount"),

        # Calculate gross amount
        (
            col("s.quantity") * col("s.unit_price")
        ).cast("decimal(14,2)").alias("gross_amount"),

        # Calculate net amount
        (
            (col("s.quantity") * col("s.unit_price"))
            - col("s.discount")
        ).cast("decimal(14,2)").alias("net_amount")
    )
)

display(df_fact)

In [0]:
from pyspark.sql.functions import count, countDistinct, sum, when

print("Total Fact Records:", df_fact.count())

print(
    "Unique Orders:",
    df_fact.select("order_id").distinct().count()
)

display(
    df_fact.select(
        count(when(col("date_key").isNull(), True)).alias("Missing Date Keys"),
        count(when(col("customer_key").isNull(), True)).alias("Missing Customer Keys"),
        count(when(col("product_key").isNull(), True)).alias("Missing Product Keys"),
        count(when(col("payment_key").isNull(), True)).alias("Missing Payment Keys")
    )
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS final_assesment.gold.fact_orders (
    order_id STRING NOT NULL,

    date_key INT,
    customer_key BIGINT,
    product_key BIGINT,
    payment_key BIGINT,

    order_status STRING,
    order_updated_at TIMESTAMP,

    quantity INT,
    unit_price DECIMAL(12,2),
    discount DECIMAL(12,2),

    gross_amount DECIMAL(14,2),
    net_amount DECIMAL(14,2),

    CONSTRAINT pk_fact_orders
        PRIMARY KEY (order_id)
)
USING DELTA;

In [0]:
fact_target = DeltaTable.forName(
    spark,
    "final_assesment.gold.fact_orders"
)

(
    fact_target.alias("t")
    .merge(
        df_fact.alias("s"),
        "t.order_id = s.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Fact table merged successfully!")

In [0]:
%sql
SELECT *
FROM final_assesment.gold.fact_orders
ORDER BY order_id;

In [0]:
%sql
SELECT
    'dim_customer' AS table_name,
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_id) AS unique_records
FROM final_assesment.gold.dim_customer

UNION ALL

SELECT
    'dim_product',
    COUNT(*),
    COUNT(DISTINCT product_id)
FROM final_assesment.gold.dim_product

UNION ALL

SELECT
    'dim_payment',
    COUNT(*),
    COUNT(DISTINCT payment_method_id)
FROM final_assesment.gold.dim_payment

UNION ALL

SELECT
    'fact_orders',
    COUNT(*),
    COUNT(DISTINCT order_id)
FROM final_assesment.gold.fact_orders;